# TDA of Consciousness States: Testing the Geometric Qualia Hypothesis

**SCPN Claim (Paper 10, L10)**: Qualia ARE geometric properties of neural
state spaces. Specifically, the Betti numbers of the state-space manifold
should change at the boundary between conscious and unconscious states.

**Test**: Simulate multi-oscillator dynamics at varying K (coupling strength).
Compute persistent homology (H0, H1, H2) of the phase-space point cloud.
If the SCPN is correct, there should be a topological phase transition
at K_c where H1 (loops = information circuits) appears or vanishes.

**Why this matters**: IIT uses Phi, GNW uses ignition, Orch-OR uses
quantum gravity. The SCPN predicts a TOPOLOGICAL signature. If Betti
numbers correlate with consciousness states, that's a new observable
that no other framework predicts.

In [ ]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.stats import pearsonr

from scpn_quantum_control.bridge import build_knm_paper27, OMEGA_N_16

## 1. Persistent Homology via Vietoris-Rips (pure NumPy)

No ripser/giotto dependency. We compute a filtration by distance threshold
and track connected components (H0) and loops (H1) via union-find and
cycle detection on the simplicial complex.

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.n_components = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False  # already connected = loop created
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        self.n_components -= 1
        return True  # merged


def persistent_homology_h0_h1(points, max_eps=None, n_steps=50):
    """Compute H0 (components) and H1 (loops) persistence from point cloud.
    
    H0: birth=0 for all points, death=epsilon when merged.
    H1: birth=epsilon when a cycle forms (edge added between already-connected nodes),
        death approximated by next threshold where a triangle fills the cycle.
    """
    n = len(points)
    D = squareform(pdist(points))
    if max_eps is None:
        max_eps = np.percentile(D[D > 0], 95)
    
    # Sort all edges by distance
    edges = []
    for i in range(n):
        for j in range(i+1, n):
            if D[i,j] <= max_eps:
                edges.append((D[i,j], i, j))
    edges.sort()
    
    uf = UnionFind(n)
    h0_deaths = []  # epsilon values where components merge
    h1_births = []  # epsilon values where cycles form
    
    for eps, i, j in edges:
        merged = uf.union(i, j)
        if merged:
            h0_deaths.append(eps)
        else:
            h1_births.append(eps)
    
    # H0 persistence: long-lived components
    h0_persistence = sorted(h0_deaths, reverse=True)  # longest-lived first
    
    # H1: number of independent cycles at each threshold
    thresholds = np.linspace(0, max_eps, n_steps)
    h0_curve = []
    h1_curve = []
    for eps in thresholds:
        uf_t = UnionFind(n)
        n_edges = 0
        for d, i, j in edges:
            if d <= eps:
                uf_t.union(i, j)
                n_edges += 1
        # Euler characteristic: V - E + F = chi = b0 - b1 + b2
        # For 1-skeleton (no triangles): b1 = n_edges - n + b0
        b0 = uf_t.n_components
        b1 = max(0, n_edges - n + b0)
        h0_curve.append(b0)
        h1_curve.append(b1)
    
    return {
        'thresholds': thresholds,
        'h0': np.array(h0_curve),
        'h1': np.array(h1_curve),
        'h0_persistence': h0_persistence[:5],
        'h1_birth_count': len(h1_births),
        'max_h1': max(h1_curve) if h1_curve else 0,
    }

## 2. Kuramoto Phase Space at Varying K

Simulate N=8 oscillators. Collect phase trajectories as point clouds
in the N-dimensional phase torus. Compute TDA at each K value.

In [ ]:
def simulate_kuramoto_trajectory(N, K_matrix, omega, T=20.0, dt=0.01, 
                                  noise=0.05, seed=42, n_samples=200):
    """Return phase-space point cloud from Kuramoto dynamics."""
    rng = np.random.default_rng(seed)
    n_steps = int(T / dt)
    phases = rng.uniform(0, 2*np.pi, N)
    
    # Collect samples after transient
    transient = n_steps // 2
    sample_interval = max(1, (n_steps - transient) // n_samples)
    points = []
    
    for t in range(n_steps):
        dphi = omega.copy()
        for i in range(N):
            for j in range(N):
                dphi[i] += K_matrix[i,j] * np.sin(phases[j] - phases[i])
        phases += dt * dphi + np.sqrt(dt) * noise * rng.normal(size=N)
        phases = phases % (2 * np.pi)
        
        if t >= transient and (t - transient) % sample_interval == 0:
            # Embed on torus: (cos(phi), sin(phi)) for each oscillator
            point = np.concatenate([np.cos(phases), np.sin(phases)])
            points.append(point)
    
    return np.array(points[:n_samples])


N = 8
K_base = build_knm_paper27(L=N)
omega = OMEGA_N_16[:N]

# Sweep K from subcritical to supercritical
K_scales = [0.0, 0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]

print(f'Simulating {N}-oscillator Kuramoto at {len(K_scales)} coupling strengths...')
print(f'{"K_scale":>8} {"R_mean":>8} {"max_H1":>8} {"H1_births":>10} {"H0_gap":>8}  State')
print('-' * 62)

results = []
for k_scale in K_scales:
    K = K_base * k_scale
    points = simulate_kuramoto_trajectory(N, K, omega, T=30.0, n_samples=150)
    
    # Order parameter
    # Reconstruct phases from embedding
    cos_phi = points[:, :N]
    sin_phi = points[:, N:]
    R_vals = np.abs(np.mean(cos_phi + 1j * sin_phi, axis=1))
    R_mean = np.mean(R_vals)
    
    # TDA
    tda = persistent_homology_h0_h1(points, n_steps=40)
    
    # H0 persistence gap: ratio of longest to second-longest H0 bar
    h0_gap = 0.0
    if len(tda['h0_persistence']) >= 2 and tda['h0_persistence'][1] > 0:
        h0_gap = tda['h0_persistence'][0] / tda['h0_persistence'][1]
    
    state = 'DESYNC' if R_mean < 0.4 else ('PARTIAL' if R_mean < 0.7 else 'SYNC')
    
    print(f'{k_scale:8.1f} {R_mean:8.3f} {tda["max_h1"]:8d} {tda["h1_birth_count"]:10d} {h0_gap:8.2f}  {state}')
    results.append({
        'K_scale': k_scale,
        'R_mean': R_mean,
        'max_h1': tda['max_h1'],
        'h1_births': tda['h1_birth_count'],
        'h0_gap': h0_gap,
        'state': state,
    })

In [ ]:
# Test the Geometric Qualia Hypothesis
R_vals = [r['R_mean'] for r in results]
h1_vals = [r['max_h1'] for r in results]
h1_birth_vals = [r['h1_births'] for r in results]

# Does H1 correlate with consciousness (R > 0.4)?
r_h1_R, p_h1_R = pearsonr(R_vals, h1_vals) if len(set(h1_vals)) > 1 else (0, 1)
r_births_R, p_births_R = pearsonr(R_vals, h1_birth_vals) if len(set(h1_birth_vals)) > 1 else (0, 1)

print('\n=== GEOMETRIC QUALIA HYPOTHESIS TEST ===')
print(f'Correlation R vs max_H1:      r={r_h1_R:.4f}, p={p_h1_R:.4f}')
print(f'Correlation R vs H1_births:   r={r_births_R:.4f}, p={p_births_R:.4f}')
print()

# The SCPN predicts:
# 1. In desync (unconscious): low H1, phase space is a diffuse cloud
# 2. At K_c (transition): H1 peaks, topology is richest
# 3. In sync (conscious): H1 stabilises, point cloud collapses to low-dim manifold
print('SCPN PREDICTION:')
print('  Desync (K~0): diffuse cloud, H1 ~ noise')
print('  Transition (K~K_c): PEAK H1 (richest topology)')
print('  Sync (K>>K_c): collapsed manifold, H1 drops')
print()

# Check for non-monotonic H1 (peak at intermediate K)
h1_peak_idx = np.argmax(h1_vals)
h1_peak_K = K_scales[h1_peak_idx]
print(f'H1 peaks at K_scale = {h1_peak_K}')
if 0 < h1_peak_idx < len(K_scales) - 1:
    print('  NON-MONOTONIC: H1 peaks at intermediate coupling')
    print('  This is the topological signature of the sync transition')
    print('  -> CONSISTENT with Geometric Qualia Hypothesis')
elif h1_peak_idx == 0:
    print('  H1 peaks at lowest K (noise-dominated topology)')
    print('  -> Topology reflects noise, not dynamics')
else:
    print('  H1 peaks at highest K (monotonic increase)')
    print('  -> More coupling = more loops (trivial)')

In [ ]:
# Compare to IIT Phi (approximate)
# Use pairwise MI summed over all oscillator pairs (avoids high-dim histogramdd)
from scipy.stats import entropy as scipy_entropy

def pairwise_mi(points, n_bins=20):
    """Sum of pairwise MI across all dimension pairs."""
    d = points.shape[1]
    total_mi = 0.0
    for i in range(d):
        for j in range(i+1, d):
            h_xy, _, _ = np.histogram2d(points[:, i], points[:, j], bins=n_bins)
            h_xy = h_xy / h_xy.sum()
            h_x = h_xy.sum(axis=1)
            h_y = h_xy.sum(axis=0)
            H_X = scipy_entropy(h_x + 1e-12)
            H_Y = scipy_entropy(h_y + 1e-12)
            H_XY = scipy_entropy(h_xy.ravel() + 1e-12)
            total_mi += max(0, H_X + H_Y - H_XY)
    return total_mi

print('\n=== COMPARISON: SCPN TOPOLOGY vs IIT PHI (approximate) ===')
print(f'{"K_scale":>8} {"R":>6} {"max_H1":>7} {"MI_pairs":>10}  {"Verdict":>12}')
print('-' * 55)

for res in results:
    k = res['K_scale']
    K = K_base * k
    points = simulate_kuramoto_trajectory(N, K, omega, T=20.0, n_samples=100)
    MI = pairwise_mi(points)
    
    verdict = 'H1 better' if res['max_h1'] > 0 and MI < 0.5 else (
              'MI better' if MI > 0.5 and res['max_h1'] == 0 else 'similar')
    
    print(f'{k:8.1f} {res["R_mean"]:6.3f} {res["max_h1"]:7d} {MI:10.4f}  {verdict:>12}')

print()
print('KEY QUESTION: Does H1 detect the consciousness boundary')
print('more sharply than pairwise MI? If yes, topology is a better observable.')

In [ ]:
import json
print('--- JSON RESULTS ---')
print(json.dumps({
    'n_oscillators': N,
    'K_scales': K_scales,
    'R_values': [round(r['R_mean'], 4) for r in results],
    'max_H1': [r['max_h1'] for r in results],
    'H1_births': [r['h1_births'] for r in results],
    'r_H1_vs_R': round(r_h1_R, 4),
    'p_H1_vs_R': round(p_h1_R, 4),
    'H1_peak_K': h1_peak_K,
    'hypothesis': 'Geometric Qualia: Betti numbers change at consciousness boundary',
}, indent=2))